In [ ]:
# ============================================================
# 0. INSTALL DEPENDENCIES
# ============================================================
# Pinned to the Ultralytics version used across this project so that
# metric behavior, model-config parsing and exports stay comparable.

!pip -q install "ultralytics==8.4.173" "scikit-learn>=1.4" "ImageHash>=4.3" pyyaml tqdm

import ultralytics, torch
print("Ultralytics:", ultralytics.__version__, "| PyTorch:", torch.__version__)
assert ultralytics.__version__ == "8.4.173", "Pin ultralytics to 8.4.173 (notebook code targets this version)"
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not available. In Colab choose Runtime > Change runtime type > GPU, "
        "then reconnect your VS Code Colab session."
    )
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 1. MOUNT DRIVE + CONFIGURATION
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

# ------------------------------
# Dataset location
# ------------------------------
DATASET_ROOT = Path("/content/drive/MyDrive/ganoderma")

if (DATASET_ROOT / "valid").exists():
    VAL_FOLDER_NAME = "valid"
elif (DATASET_ROOT / "val").exists():
    VAL_FOLDER_NAME = "val"
else:
    raise FileNotFoundError(
        f"Could not find {DATASET_ROOT / 'valid'} or {DATASET_ROOT / 'val'}"
    )

# ------------------------------
# Experiment output (persistent, on Drive)
# ------------------------------
EXP_NAME = "leak_free_yolov8s_lite_cv_v1"
EXP_ROOT = DATASET_ROOT / "experiments" / EXP_NAME
EXP_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_DATASET = Path("/content/ganoderma_source")     # fast local copy of original data
LOCAL_WORK    = Path("/content/ganoderma_cv_work")    # local training workspace

# ------------------------------
# Training / reproducibility
# ------------------------------
SEED      = 42
N_FOLDS   = 5
EPOCHS    = 50
IMGSZ     = 640
DEVICE    = 0          # GPU 0
PATIENCE  = 20
WORKERS   = 2

# ------------------------------
# Models to train in this CV
#   cfg: "name.pt" = COCO-pretrained reference (auto-downloaded)
#        "name.yaml" = custom architecture trained from scratch
# NOTE: the Lite architectures have no pretrained weights, so they train from
# scratch, while the reference models follow the existing project methodology
# (COCO-pretrained start) and represent the deployed YOLOv8n baseline.
# ------------------------------
MODELS = {
    "yolov8n": {
        "cfg": "yolov8n.pt",
        "pretrained": True,
        "enabled": True,      # baseline reference
        "batch": 32,
    },
    "yolov8s": {
        "cfg": "yolov8s.pt",
        "pretrained": True,
        "enabled": False,     # optional reference; roughly doubles CV time
        "batch": 30,
    },
    "yolov8s_lite": {
        "cfg": "yolov8s_lite.yaml",       # written to LOCAL_WORK/models below
        "pretrained": False,
        "enabled": False,
        "batch": 16,
    },
    "yolov8s_lite_se": {
        "cfg": "yolov8s_lite_se.yaml",
        "pretrained": False,
        "enabled": False,
        "batch": 16,
    },
}

# The spec architecture uses depth_multiple = 1.0 (exact C2f repeats 2/4/4/2
# in the backbone, 2 in the neck -> 35.3 M params / 102.1 GFLOPs at nc=1).
# Smaller variants: 0.5 -> 25.5 M / 72.9 G   |   0.33 -> 24.4 M / 66.3 G.
# If you change this here, keep the SAME value in notebook 2.
LITE_DEPTH_MULTIPLE = 1.0

# ------------------------------
# Experiment switches
# ------------------------------
FOLDS_TO_RUN = list(range(1, N_FOLDS + 1))

# Skip a model/fold when best.pt + metrics.csv already exist on Drive.
RESUME_COMPLETED = True
FORCE_RETRAIN    = False

# Rebuild split manifests even if they already exist.
FORCE_REBUILD_SPLITS = False

# Keep local fold workdirs after training. False saves Colab disk space.
KEEP_LOCAL_FOLD_WORKDIRS = False

# Reuse the leak-free audit/splits of the previous experiment when available
# (identical data + seed => identical folds; saves ~15-20 min per session).
REUSE_PREV_AUDIT_SPLITS = True
PREV_EXP = "leak_free_yolov8s_cv_v1"

# Dataset audit behavior (same as the existing notebook):
STRICT_EXACT_TEST_LEAKAGE = True
PHASH_WARN_DISTANCE = 4

# ------------------------------
# Quick smoke test
# ------------------------------
SMOKE_TEST = False
if SMOKE_TEST:
    EPOCHS = 1
    FOLDS_TO_RUN = [1]

print("Dataset root :", DATASET_ROOT)
print("Validation   :", VAL_FOLDER_NAME)
print("Experiment   :", EXP_ROOT)
print("Folds        :", FOLDS_TO_RUN)
print("Epochs       :", EPOCHS)
print("Depth multiple:", LITE_DEPTH_MULTIPLE)
print("Models       :", [k for k, v in MODELS.items() if v["enabled"]])

In [ ]:
# ============================================================
# 2. LITESE MODULE + ARCHITECTURE YAMLS + SANITY CHECK (spec Part 9)
# ============================================================
import torch
import torch.nn as nn
import yaml
import numpy as np
import pandas as pd
import re, io, logging, contextlib

# ------------------------------------------------------------
# LiteSE — lightweight Squeeze-and-Excitation channel attention
#   input [B, C, H, W] -> output [B, C, H, W]  (shape preserved)
#   reduction 8: C -> C//8 -> C   (768 -> 96 -> 768)
# Ops (all ONNX-standard, QAIRT/QNN-friendly):
#   GlobalAveragePool, Conv2d 1x1, ReLU, Sigmoid, Mul
# No BatchNorm, no transformer attention, no custom/inplace operators.
# ------------------------------------------------------------
class LiteSE(nn.Module):
    # Accepts both call patterns produced by Ultralytics parse_model():
    #   generic-module path (fresh install): LiteSE(768, 8)
    #   base-module path (if registered)   : LiteSE(c1, c2, 8)
    def __init__(self, c1: int, reduction: int = 8, *extra):
        if extra:
            # parse_model injected a c2 channel argument (c1, c2, reduction).
            # c2 == c1 here because width_multiple=1.0 and channel counts are
            # explicit in the YAML, so only the trailing reduction is needed.
            reduction = extra[0]
        super().__init__()
        c_mid = max(4, c1 // reduction)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),   # -> GlobalAveragePool in ONNX
            nn.Conv2d(c1, c_mid, kernel_size=1, stride=1, padding=0, bias=True),
            nn.ReLU(),                 # inplace=False -> clean ONNX graph
            nn.Conv2d(c_mid, c1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.fc(x)

# Ultralytics 8.4.173 parse_model() resolves YAML module names with
# globals()[m] inside ultralytics.nn.tasks -> register the class there.
import ultralytics.nn.tasks as tasks
tasks.LiteSE = LiteSE
print("LiteSE registered in ultralytics.nn.tasks")

# ------------------------------------------------------------
# Build the two Lite YAML configs (programmatically, so the layer
# indices of the PAN path stay correct with/without the LiteSE block).
# ------------------------------------------------------------
MODELS_DIR = LOCAL_WORK / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

def build_lite_yaml(se: bool, depth_multiple: float):
    backbone = [
        [-1, 1, "Conv", [64, 3, 2]],    # 0  P1/2
        [-1, 1, "Conv", [128, 3, 2]],   # 1  P2/4
        [-1, 2, "C2f", [128, True]],    # 2
        [-1, 1, "Conv", [256, 3, 2]],   # 3  P3/8
        [-1, 4, "C2f", [256, True]],    # 4  <-- P3 / stride 8
        [-1, 1, "Conv", [384, 3, 2]],   # 5  P4/16
        [-1, 4, "C2f", [384, True]],    # 6  <-- P4 / stride 16
        [-1, 1, "Conv", [768, 3, 2]],   # 7  P5/32
        [-1, 2, "C2f", [768, True]],    # 8
        [-1, 1, "SPPF", [768, 5]],      # 9
    ]
    if se:
        backbone.append([-1, 1, "LiteSE", [768, 8]])   # 10 channel attention
    h0 = 10 + (1 if se else 0)  # index of the first head layer
    head = [
        [-1, 1, "Conv", [384, 1, 1]],                    # h0+0  P5: 768 -> 384
        [-1, 1, "nn.Upsample", [None, 2, "nearest"]],    # h0+1  x2
        [[-1, 6], 1, "Concat", [1]],                     # h0+2  cat P4: 384+384=768
        [-1, 2, "C2f", [384, True]],                     # h0+3  P4/16
        [-1, 1, "Conv", [192, 1, 1]],                    # h0+4  384 -> 192
        [-1, 1, "nn.Upsample", [None, 2, "nearest"]],    # h0+5  x2
        [[-1, 4], 1, "Concat", [1]],                     # h0+6  cat P3: 192+256=448
        [-1, 2, "C2f", [192, True]],                     # h0+7  P3/8
        [-1, 1, "Conv", [192, 3, 2]],                    # h0+8  PAN P3 -> P4
        [[-1, h0 + 3], 1, "Concat", [1]],                # h0+9  cat P4: 192+384=576
        [-1, 2, "C2f", [384, True]],                     # h0+10 P4/16
        [-1, 1, "Conv", [384, 3, 2]],                    # h0+11 PAN P4 -> P5
        [[-1, 10 if se else 9], 1, "Concat", [1]],       # h0+12 cat P5: 384+768=1152
        [-1, 2, "C2f", [768, True]],                     # h0+13 P5/32
        [[h0 + 7, h0 + 10, h0 + 13], 1, "Detect", ["nc"]],
    ]
    return {
        "nc": 1,
        "depth_multiple": depth_multiple,
        "width_multiple": 1.0,
        "backbone": backbone,
        "head": head,
    }

LITE_FILES = {}
for se, name in [(False, "yolov8s_lite"), (True, "yolov8s_lite_se")]:
    p = MODELS_DIR / f"{name}.yaml"
    with p.open("w") as f:
        yaml.safe_dump(build_lite_yaml(se, LITE_DEPTH_MULTIPLE), f)
    LITE_FILES[name] = str(p)
    print("wrote", p)

# Also write nc=1 versions of the official yolov8n/s YAMLs, used only to
# report exact params/GFLOPs for the comparison table (nc=1 detection head).
import ultralytics
PKG_YAML = Path(ultralytics.__file__).parent / "cfg" / "models" / "v8" / "yolov8.yaml"
base_v8 = yaml.safe_load(PKG_YAML.read_text())
REF_NC1_FILES = {}
for scale, name in [("n", "yolov8n"), ("s", "yolov8s")]:
    d = yaml.safe_load(yaml.dump(base_v8))
    d["nc"] = 1
    d["scale"] = scale
    p = MODELS_DIR / f"{name}_nc1.yaml"
    with p.open("w") as f:
        yaml.safe_dump(d, f)
    REF_NC1_FILES[name] = str(p)
    print("wrote", p)

# ------------------------------------------------------------
# Helper: capture params/GFLOPs via the ultralytics model_info printer
# ------------------------------------------------------------
ANSI = re.compile(r"\x1b\[[0-9;]*m")

def capture_gflops(model, imgsz=IMGSZ):
    from ultralytics.utils.torch_utils import model_info
    buf = io.StringIO()
    lg = logging.getLogger("ultralytics")
    h = logging.StreamHandler(buf)
    lg.addHandler(h)
    try:
        model_info(model, imgsz=imgsz, verbose=True)
    finally:
        lg.removeHandler(h)
    txt = ANSI.sub("", buf.getvalue())
    m = re.search(r"([\d.]+)\s+GFLOPs", txt)
    return float(m.group(1)) if m else float("nan")

# ------------------------------------------------------------
# Instantiate every enabled model, print info, run a dummy forward
# pass, check shapes and NaN/Inf (spec Part 9: torch.randn(1,3,640,640)).
# ------------------------------------------------------------
from ultralytics import YOLO

ARCH_SUMMARY = {}
for name, mcfg in MODELS.items():
    if not mcfg["enabled"]:
        continue
    print("\n" + "=" * 80)
    print(f"MODEL: {name}")
    print("=" * 80)

    if mcfg["pretrained"]:
        m = YOLO(mcfg["cfg"])              # downloads COCO weights
    else:
        m = YOLO(LITE_FILES[name])         # builds from scratch

    m.info(detailed=False)

    mm = m.model
    x = torch.randn(1, 3, IMGSZ, IMGSZ)

    # training-mode forward -> per-level feature maps
    mm.train()
    with torch.no_grad():
        out_train = mm(x)
    feats = None
    if isinstance(out_train, dict):
        feats = out_train.get("feats")
        if feats is None:
            for v in out_train.values():
                if isinstance(v, (list, tuple)):
                    feats = v
                    break
    elif isinstance(out_train, (list, tuple)):
        feats = out_train
    print("per-level feature shapes:", [tuple(t.shape) for t in feats])
    if name.startswith("yolov8s_lite"):
        assert [tuple(t.shape)[1] for t in feats] == [192, 384, 768], "head channel check failed"
        assert [tuple(t.shape)[2] for t in feats] == [80, 40, 20], "stride check failed"

    # inference-mode forward -> standard YOLO output
    mm.eval()
    with torch.no_grad():
        out_inf = mm(x)
    if isinstance(out_inf, (list, tuple)):
        out_inf = out_inf[0]
    nc_m = int(mm.model[-1].nc)
    print("inference output shape:", tuple(out_inf.shape), f"(nc={nc_m})")
    assert tuple(out_inf.shape) == (1, 4 + nc_m, 8400), "unexpected inference output shape"

    ok = bool(torch.isfinite(out_inf).all())
    print("finite:", ok, "| NaN:", bool(torch.isnan(out_inf).any()), "| Inf:", bool(torch.isinf(out_inf).any()))
    assert ok, "non-finite outputs detected"

    n_params = int(sum(p.numel() for p in mm.parameters()))
    gflops = capture_gflops(mm)
    ARCH_SUMMARY[name] = {
        "model": name,
        "cfg": mcfg["cfg"],
        "pretrained": mcfg["pretrained"],
        "params": n_params,
        "gflops": gflops,
        "layers": len(mm.model),
    }
    print(f"{name}: params={n_params/1e6:.3f}M  GFLOPs={gflops:.1f}  layers={len(mm.model)}")
    del m, mm, x
    torch.cuda.empty_cache()

# exact nc=1 params/GFLOPs for the reference models (overwrites the COCO
# nc=80 values above so the comparison table uses the trained nc=1 config)
for name in ["yolov8n", "yolov8s"]:
    m = YOLO(REF_NC1_FILES[name])
    n_params = int(sum(p.numel() for p in m.model.parameters()))
    gflops = capture_gflops(m.model)
    ARCH_SUMMARY[name] = {
        "model": name,
        "cfg": MODELS[name]["cfg"],
        "pretrained": True,
        "params": n_params,
        "gflops": gflops,
        "layers": len(m.model.model),
    }
    print(f"reference nc=1: {name}: params={n_params/1e6:.3f}M GFLOPs={gflops:.1f}")

arch_df = pd.DataFrame(list(ARCH_SUMMARY.values()))
(EXP_ROOT / "models").mkdir(parents=True, exist_ok=True)
arch_df.to_csv(EXP_ROOT / "models" / "architecture_summary.csv", index=False)
print("\nARCHITECTURE SUMMARY (saved to Drive)")
print(arch_df.round(2).to_string(index=False))

# LiteSE standalone unit check
blk = LiteSE(768, 8)
with torch.no_grad():
    o = blk(torch.randn(2, 768, 20, 20))
assert o.shape == (2, 768, 20, 20) and bool(torch.isfinite(o).all())
print("\nLiteSE standalone check: shape preserved (2,768,20,20) -> OK, finite: OK")

In [ ]:
# ============================================================
# 3. IMPORTS, SEEDS, METRIC HELPERS
# ============================================================
import os
import gc
import cv2
import json
import math
import time
import shutil
import random
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import imagehash
from tqdm.auto import tqdm

import torch
from ultralytics import YOLO
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore", category=UserWarning)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def metric_value(res, name):
    # Version-tolerant accessor for Ultralytics detection metrics.
    v = getattr(res.box, name)
    if callable(v):
        v = v()
    return float(v)

def extract_metrics(res):
    p = metric_value(res, "mp")
    r = metric_value(res, "mr")
    map50 = metric_value(res, "map50")
    map5095 = metric_value(res, "map")
    f1 = 2 * p * r / (p + r + 1e-12)
    return {"mAP50": map50, "mAP50-95": map5095, "Precision": p, "Recall": r, "F1": f1}

def write_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)

def sync_tree(src, dst):
    # Replace dst with a copy of src.
    src, dst = Path(src), Path(dst)
    if dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)

def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

config_snapshot = {
    "seed": SEED, "n_folds": N_FOLDS, "epochs": EPOCHS, "imgsz": IMGSZ,
    "device": DEVICE, "patience": PATIENCE, "workers": WORKERS,
    "models": {k: {kk: vv for kk, vv in v.items() if kk != "batch"} for k, v in MODELS.items()},
    "lite_depth_multiple": LITE_DEPTH_MULTIPLE,
    "folds_to_run": FOLDS_TO_RUN,
    "ultralytics_version": ultralytics.__version__,
    "torch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
write_json(config_snapshot, EXP_ROOT / "experiment_config.json")
print("config snapshot saved")

In [ ]:
# ============================================================
# 4. CACHE ORIGINAL DATA LOCALLY + DATASET AUDIT
# ============================================================
# Drive is persistent but slow for thousands of small reads: copy the
# original train/valid/test data to /content for speed. The original
# dataset folders on Drive are never modified.
# The audit (sha256 + pHash + label checks) is reused from the previous
# experiment when the counts match, otherwise it is rebuilt.

SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def copy_split_to_local(src_split_name, dst_split_name):
    src = DATASET_ROOT / src_split_name
    dst = LOCAL_DATASET / dst_split_name
    if not src.exists():
        raise FileNotFoundError(src)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

LOCAL_DATASET.mkdir(parents=True, exist_ok=True)
print("Caching original dataset locally...")
copy_split_to_local("train", "train")
copy_split_to_local(VAL_FOLDER_NAME, "valid")
copy_split_to_local("test", "test")
print("local cache ready:", LOCAL_DATASET)

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def label_signature(label_path):
    if not label_path.exists():
        return ""
    lines = []
    with open(label_path, "r") as f:
        for line in f:
            s = " ".join(line.strip().split())
            if s:
                lines.append(s)
    return "|".join(sorted(lines))

def audit_label(label_path):
    issues = []
    boxes = []
    if not label_path.exists():
        return boxes, issues
    with open(label_path, "r") as f:
        for line_no, line in enumerate(f, start=1):
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) != 5:
                issues.append(f"line {line_no}: expected 5 values, got {len(parts)}")
                continue
            try:
                cls, x, y, w, h = map(float, parts)
            except Exception:
                issues.append(f"line {line_no}: non-numeric value")
                continue
            if int(cls) != 0:
                issues.append(f"line {line_no}: class must be 0, got {cls}")
            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                issues.append(f"line {line_no}: invalid normalized box")
            boxes.append((int(cls), x, y, w, h))
    return boxes, issues

def build_dataset_index():
    rows = []
    issues = []
    for split in ["train", "valid", "test"]:
        img_dir = LOCAL_DATASET / split / "images"
        lbl_dir = LOCAL_DATASET / split / "labels"
        if not img_dir.exists() or not lbl_dir.exists():
            raise FileNotFoundError(f"{img_dir} or {lbl_dir} missing")
        image_paths = sorted(
            p for p in img_dir.iterdir()
            if p.is_file() and p.suffix.lower() in SUPPORTED_EXTS
        )
        for img_path in tqdm(image_paths, desc=f"Audit {split}"):
            label_path = lbl_dir / f"{img_path.stem}.txt"
            img = cv2.imread(str(img_path))
            if img is None:
                issues.append({"uid": f"{split}::{img_path.name}", "issue": "image cannot be read"})
                continue
            boxes, label_issues = audit_label(label_path)
            for item in label_issues:
                issues.append({"uid": f"{split}::{img_path.name}", "issue": item})
            try:
                ph = str(imagehash.phash(Image.open(img_path).convert("RGB")))
            except Exception as e:
                ph = ""
                issues.append({"uid": f"{split}::{img_path.name}", "issue": f"pHash failed: {e}"})
            rows.append({
                "uid": f"{split}::{img_path.name}",
                "source_split": split,
                "filename": img_path.name,
                "rel_image": f"{split}/images/{img_path.name}",
                "rel_label": f"{split}/labels/{img_path.stem}.txt",
                "n_boxes": len(boxes),
                "positive": int(len(boxes) > 0),
                "sha256": sha256_file(img_path),
                "phash": ph,
                "label_signature": label_signature(label_path),
                "height": img.shape[0],
                "width": img.shape[1],
            })
    return pd.DataFrame(rows), pd.DataFrame(issues)

AUDIT_DIR = EXP_ROOT / "audits"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

PREV_AUDIT_CSV = DATASET_ROOT / "experiments" / PREV_EXP / "audits" / "dataset_index.csv"

def local_image_count(split_dir):
    d = LOCAL_DATASET / split_dir / "images"
    if not d.exists():
        return -1
    return len([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED_EXTS])

index_df = None
if REUSE_PREV_AUDIT_SPLITS and PREV_AUDIT_CSV.exists():
    cand = pd.read_csv(PREV_AUDIT_CSV)
    ok = True
    for split, name in [("train", "train"), ("valid", "valid"), ("test", "test")]:
        n_idx = int((cand["source_split"] == split).sum())
        n_loc = local_image_count(name)
        if n_idx != n_loc:
            ok = False
            print(f"  audit mismatch {split}: audit={n_idx} local={n_loc}")
    if ok:
        index_df = cand
        print("Reused dataset audit/index from previous experiment:", PREV_AUDIT_CSV)

if index_df is None:
    print("Building fresh dataset audit (sha256 + pHash + labels)...")
    index_df, issue_df = build_dataset_index()
    index_df["leak_group"] = np.where(
        index_df["phash"].astype(str).str.len() > 0,
        "phash:" + index_df["phash"].astype(str),
        "sha256:" + index_df["sha256"].astype(str),
    )
    index_df.to_csv(AUDIT_DIR / "dataset_index.csv", index=False)
    issue_df.to_csv(AUDIT_DIR / "label_issues.csv", index=False)
    if len(issue_df):
        print(f"Label/image audit found {len(issue_df)} issue(s):")
        print(issue_df.head(20).to_string(index=False))
        raise RuntimeError(f"Fix the audit issues in {AUDIT_DIR / 'label_issues.csv'} before training.")
else:
    index_df = index_df.copy()
    index_df["leak_group"] = np.where(
        index_df["phash"].astype(str).str.len() > 0,
        "phash:" + index_df["phash"].astype(str),
        "sha256:" + index_df["sha256"].astype(str),
    )
    index_df.to_csv(AUDIT_DIR / "dataset_index.csv", index=False)

print("\nDataset counts:")
print(index_df.groupby("source_split").agg(
    images=("uid", "count"),
    positive=("positive", "sum"),
    boxes=("n_boxes", "sum"),
).to_string())

# Exact-duplicate / leakage checks (same as the existing notebook)
dev_df_all = index_df[index_df["source_split"].isin(["train", "valid"])].copy()
test_df_all = index_df[index_df["source_split"] == "test"].copy()

exact_dev_test_overlap = sorted(set(dev_df_all["sha256"]) & set(test_df_all["sha256"]))
print(f"\nExact dev<->independent-test hash overlap: {len(exact_dev_test_overlap)}")
if exact_dev_test_overlap:
    overlap_df = index_df[index_df["sha256"].isin(exact_dev_test_overlap)].copy()
    overlap_df.to_csv(AUDIT_DIR / "EXACT_DEV_TEST_LEAKAGE.csv", index=False)
    msg = "Exact duplicate images exist between train/valid and the original test set."
    if STRICT_EXACT_TEST_LEAKAGE:
        raise RuntimeError(msg)
    else:
        print("WARNING:", msg)

dev_ph = [(r.uid, imagehash.hex_to_hash(r.phash)) for r in dev_df_all.itertuples()
          if isinstance(r.phash, str) and r.phash]
test_ph = [(r.uid, imagehash.hex_to_hash(r.phash)) for r in test_df_all.itertuples()
           if isinstance(r.phash, str) and r.phash]
near_rows = []
for test_uid, test_hash in tqdm(test_ph, desc="pHash test audit"):
    for dev_uid, dev_hash in dev_ph:
        d = test_hash - dev_hash
        if d <= PHASH_WARN_DISTANCE:
            near_rows.append({"test_uid": test_uid, "dev_uid": dev_uid, "phash_distance": int(d)})
near_df = pd.DataFrame(near_rows)
near_df.to_csv(AUDIT_DIR / "near_duplicate_candidates_dev_vs_test.csv", index=False)
if len(near_df):
    print(f"Found {len(near_df)} dev<->test perceptual near-duplicate candidate pair(s) "
          f"(pHash distance <= {PHASH_WARN_DISTANCE}). Inspect: "
          f"{AUDIT_DIR / 'near_duplicate_candidates_dev_vs_test.csv'}")
else:
    print("No dev<->test pHash candidates within the warning threshold")

print("\nDataset audit complete")

In [ ]:
# ============================================================
# 5. CREATE / REUSE 5-FOLD SPLITS
# ============================================================
# - Original test/ is NOT used here.
# - CV uses only original train/ + valid/ images.
# - pHash groups are kept inside one partition (no cross-split duplicates).
# - Outer StratifiedGroupKFold (seed 42) -> held-out fold; the remaining 80%
#   is split again (seed 42+fold) into train/validation.

SPLITS_DIR = EXP_ROOT / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

split_files_expected = [
    SPLITS_DIR / f"fold_{f}_{part}.csv"
    for f in range(1, N_FOLDS + 1)
    for part in ["train", "val", "test"]
]

def split_files_exist():
    return all(p.exists() for p in split_files_expected)

def fold_csvs_ok(d):
    files = [d / f"fold_{f}_{part}.csv" for f in range(1, N_FOLDS + 1) for part in ["train", "val", "test"]]
    return all(p.exists() for p in files)

if FORCE_REBUILD_SPLITS or not split_files_exist():
    PREV_SPLITS_DIR = DATASET_ROOT / "experiments" / PREV_EXP / "splits"
    reused = False
    if REUSE_PREV_AUDIT_SPLITS and PREV_SPLITS_DIR.exists() and fold_csvs_ok(PREV_SPLITS_DIR):
        for p in PREV_SPLITS_DIR.glob("fold_*.csv"):
            shutil.copy2(p, SPLITS_DIR / p.name)
        summ = PREV_SPLITS_DIR / "fold_split_summary.csv"
        if summ.exists():
            shutil.copy2(summ, SPLITS_DIR / "fold_split_summary.csv")
        reused = True
        print("Reused identical leak-free split manifests from previous experiment:", PREV_SPLITS_DIR)

    if not reused:
        print("Creating new 5-fold split manifests...")
        dev_df = index_df[index_df["source_split"].isin(["train", "valid"])].copy().reset_index(drop=True)

        # Duplicate images with conflicting labels cannot be split safely.
        mixed = (dev_df.groupby("sha256")["label_signature"].nunique()
                 .reset_index(name="n_label_versions"))
        mixed = mixed[mixed["n_label_versions"] > 1]
        if len(mixed):
            bad = dev_df[dev_df["sha256"].isin(mixed["sha256"])].copy()
            bad.to_csv(AUDIT_DIR / "duplicate_images_with_conflicting_labels.csv", index=False)
            raise RuntimeError("Same exact image has conflicting labels. Fix before CV.")

        outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        fold_summary = []
        for fold, (outer_train_idx, heldout_idx) in enumerate(
            outer.split(dev_df, y=dev_df["positive"], groups=dev_df["leak_group"]), start=1
        ):
            outer_train = dev_df.iloc[outer_train_idx].reset_index(drop=True)
            heldout = dev_df.iloc[heldout_idx].reset_index(drop=True)

            inner = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + fold)
            inner_train_idx, val_idx = next(inner.split(
                outer_train, y=outer_train["positive"], groups=outer_train["leak_group"]))

            train_df = outer_train.iloc[inner_train_idx].reset_index(drop=True)
            val_df = outer_train.iloc[val_idx].reset_index(drop=True)
            test_fold_df = heldout.reset_index(drop=True)

            assert not (set(train_df["uid"]) & set(val_df["uid"]))
            assert not (set(train_df["uid"]) & set(test_fold_df["uid"]))
            assert not (set(val_df["uid"]) & set(test_fold_df["uid"]))
            assert not (set(train_df["leak_group"]) & set(val_df["leak_group"]))
            assert not (set(train_df["leak_group"]) & set(test_fold_df["leak_group"]))
            assert not (set(val_df["leak_group"]) & set(test_fold_df["leak_group"]))

            train_df.to_csv(SPLITS_DIR / f"fold_{fold}_train.csv", index=False)
            val_df.to_csv(SPLITS_DIR / f"fold_{fold}_val.csv", index=False)
            test_fold_df.to_csv(SPLITS_DIR / f"fold_{fold}_test.csv", index=False)

            fold_summary.append({
                "fold": fold,
                "train_images": len(train_df),
                "train_positive": int(train_df["positive"].sum()),
                "val_images": len(val_df),
                "val_positive": int(val_df["positive"].sum()),
                "test_images": len(test_fold_df),
                "test_positive": int(test_fold_df["positive"].sum()),
            })

        fold_summary_df = pd.DataFrame(fold_summary)
        fold_summary_df.to_csv(SPLITS_DIR / "fold_split_summary.csv", index=False)
        print(fold_summary_df.to_string(index=False))
        print("Leak-free split manifests saved to", SPLITS_DIR)

else:
    print("Reusing existing split manifests from", SPLITS_DIR)

print(pd.read_csv(SPLITS_DIR / "fold_split_summary.csv").to_string(index=False))

# Verify every development image appears exactly once as an outer held-out fold.
all_heldout = []
for fold in range(1, N_FOLDS + 1):
    df = pd.read_csv(SPLITS_DIR / f"fold_{fold}_test.csv")
    all_heldout.extend(df["uid"].tolist())

heldout_counts = pd.Series(all_heldout).value_counts()
expected_dev_uids = set(index_df[index_df["source_split"].isin(["train", "valid"])]["uid"])
assert set(all_heldout) == expected_dev_uids
assert heldout_counts.max() == 1 and heldout_counts.min() == 1
print("Every development image is held out exactly once across the 5 folds")

In [ ]:
# ============================================================
# 6. TRAIN + EVALUATE ALL ENABLED MODELS x FOLDS
# ============================================================
# For each model and fold:
#   - materialize the fold partitions locally
#   - train (pretrained .pt for references, from-scratch YAML for Lite models)
#   - reload best.pt and evaluate ONLY on that fold's true held-out test split
#   - persist best.pt + metrics + training artifacts to Google Drive
# Completed model/fold pairs are skipped when RESUME_COMPLETED=True.

def safe_materialized_name(row):
    return f"{row['source_split']}__{row['filename']}"

def record_image_path(row):
    return LOCAL_DATASET / row["rel_image"]

def record_label_path(row):
    return LOCAL_DATASET / row["rel_label"]

def materialize_original_partition(df, out_root):
    out_root = Path(out_root)
    img_dir = out_root / "images"
    lbl_dir = out_root / "labels"
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for _, row in df.iterrows():
        src_img = record_image_path(row)
        src_lbl = record_label_path(row)
        dst_img_name = safe_materialized_name(row)
        shutil.copy2(src_img, img_dir / dst_img_name)
        if src_lbl.exists():
            shutil.copy2(src_lbl, lbl_dir / f"{Path(dst_img_name).stem}.txt")
        else:
            (lbl_dir / f"{Path(dst_img_name).stem}.txt").write_text("")

def write_yolo_yaml(dataset_dir, yaml_path):
    dataset_dir = Path(dataset_dir)
    yaml_path = Path(yaml_path)
    text = (f"train: {dataset_dir / 'train' / 'images'}\n"
            f"val: {dataset_dir / 'val' / 'images'}\n"
            f"test: {dataset_dir / 'test' / 'images'}\n"
            f"nc: 1\n"
            f"names: ['Ganoderma']\n")
    yaml_path.write_text(text)

def completed_model_fold(drive_dir):
    return ((drive_dir / "best.pt").exists() and (drive_dir / "metrics.csv").exists())

def sync_run_lean(local_run_dir, drive_run_dir):
    # Persist training/eval artifacts but skip the large .pt files (already
    # copied separately as best.pt) to keep Drive usage reasonable.
    local_run_dir, drive_run_dir = Path(local_run_dir), Path(drive_run_dir)
    drive_run_dir.mkdir(parents=True, exist_ok=True)
    for item in local_run_dir.iterdir():
        if item.name == "weights":
            continue
        dst = drive_run_dir / item.name
        if dst.exists():
            if dst.is_dir():
                shutil.rmtree(dst)
            else:
                dst.unlink()
        if item.is_dir():
            shutil.copytree(item, dst)
        else:
            shutil.copy2(item, dst)

all_rows = []

for model_name, mcfg in MODELS.items():
    if not mcfg["enabled"]:
        print(f"\n[skip] {model_name} (disabled)")
        continue

    print("\n" + "#" * 90)
    print(f"MODEL: {model_name}   cfg={mcfg['cfg']}   pretrained={mcfg['pretrained']}")
    print("#" * 90)

    for fold in FOLDS_TO_RUN:
        print("\n" + "=" * 80)
        print(f"FOLD {fold}/{N_FOLDS}  ({model_name})")
        print("=" * 80)

        train_df = pd.read_csv(SPLITS_DIR / f"fold_{fold}_train.csv")
        val_df = pd.read_csv(SPLITS_DIR / f"fold_{fold}_val.csv")
        test_df = pd.read_csv(SPLITS_DIR / f"fold_{fold}_test.csv")

        # Fold-level leakage checks again immediately before training.
        assert not (set(train_df["uid"]) & set(val_df["uid"]))
        assert not (set(train_df["uid"]) & set(test_df["uid"]))
        assert not (set(val_df["uid"]) & set(test_df["uid"]))
        assert not (set(train_df["leak_group"]) & set(val_df["leak_group"]))
        assert not (set(train_df["leak_group"]) & set(test_df["leak_group"]))
        assert not (set(val_df["leak_group"]) & set(test_df["leak_group"]))

        drive_dir = EXP_ROOT / "models" / model_name / "folds" / f"fold_{fold}"
        local_dir = LOCAL_WORK / model_name / f"fold_{fold}"

        if RESUME_COMPLETED and not FORCE_RETRAIN and completed_model_fold(drive_dir):
            row = pd.read_csv(drive_dir / "metrics.csv").iloc[0].to_dict()
            all_rows.append(row)
            print(f"  {model_name} / fold {fold}: already complete, skipping "
                  f"(mAP50={row['mAP50']:.4f})")
            continue

        if local_dir.exists():
            shutil.rmtree(local_dir)
        local_dir.mkdir(parents=True, exist_ok=True)
        drive_dir.mkdir(parents=True, exist_ok=True)

        dataset_dir = local_dir / "dataset"
        print(f"  materializing partitions...")
        materialize_original_partition(train_df, dataset_dir / "train")
        materialize_original_partition(val_df, dataset_dir / "val")
        materialize_original_partition(test_df, dataset_dir / "test")

        train_count = len([p for p in (dataset_dir / "train" / "images").iterdir()
                           if p.suffix.lower() in SUPPORTED_EXTS])
        val_count = len([p for p in (dataset_dir / "val" / "images").iterdir()
                         if p.suffix.lower() in SUPPORTED_EXTS])
        test_count = len([p for p in (dataset_dir / "test" / "images").iterdir()
                          if p.suffix.lower() in SUPPORTED_EXTS])
        assert train_count == len(train_df)
        assert val_count == len(val_df)
        assert test_count == len(test_df)

        yaml_path = local_dir / "data.yaml"
        write_yolo_yaml(dataset_dir, yaml_path)
        print(f"  train={train_count}, val={val_count}, held-out={test_count}")

        seed_everything(SEED + fold)

        if mcfg["pretrained"]:
            model = YOLO(mcfg["cfg"])
        else:
            model = YOLO(LITE_FILES[model_name])

        train_result = model.train(
            data=str(yaml_path),
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=mcfg["batch"],
            device=DEVICE,
            patience=PATIENCE,
            workers=WORKERS,
            seed=SEED + fold,
            deterministic=True,
            pretrained=mcfg["pretrained"],
            project=str(local_dir),
            name="train_run",
            exist_ok=True,
            verbose=True,
            plots=True,
        )

        best_pt = local_dir / "train_run" / "weights" / "best.pt"
        if not best_pt.exists():
            raise FileNotFoundError(best_pt)

        # IMPORTANT: explicitly reload best.pt, then evaluate only this
        # fold's genuine held-out outer test fold.
        best_model = YOLO(str(best_pt))
        eval_result = best_model.val(
            data=str(yaml_path),
            split="test",
            imgsz=IMGSZ,
            batch=mcfg["batch"],
            device=DEVICE,
            project=str(local_dir),
            name="heldout_eval",
            exist_ok=True,
            plots=True,
            verbose=False,
        )

        m = extract_metrics(eval_result)
        n_params = int(sum(p.numel() for p in best_model.model.parameters()))
        gflops = capture_gflops(best_model.model)
        row = {
            "model": model_name,
            "fold": fold,
            "cfg": str(mcfg["cfg"]),
            "pretrained": bool(mcfg["pretrained"]),
            "params": n_params,
            "gflops": gflops,
            "model_size_mb": round(best_pt.stat().st_size / 1e6, 2),
            **m,
            "train_images": len(train_df),
            "val_images": len(val_df),
            "heldout_images": len(test_df),
            "epochs_requested": EPOCHS,
            "seed": SEED + fold,
        }
        pd.DataFrame([row]).to_csv(local_dir / "metrics.csv", index=False)

        # Persist to Google Drive.
        drive_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(best_pt, drive_dir / "best.pt")
        shutil.copy2(local_dir / "metrics.csv", drive_dir / "metrics.csv")
        shutil.copy2(yaml_path, drive_dir / "data.yaml")
        train_df.to_csv(drive_dir / "train_manifest.csv", index=False)
        val_df.to_csv(drive_dir / "val_manifest.csv", index=False)
        test_df.to_csv(drive_dir / "heldout_manifest.csv", index=False)
        sync_run_lean(local_dir / "train_run", drive_dir / "train_run")
        sync_run_lean(local_dir / "heldout_eval", drive_dir / "heldout_eval")

        print(f"  {model_name} / fold {fold}: "
              f"mAP50={m['mAP50']:.4f}, mAP50-95={m['mAP50-95']:.4f}, "
              f"P={m['Precision']:.4f}, R={m['Recall']:.4f}, F1={m['F1']:.4f}")

        all_rows.append(row)

        del model, best_model, train_result, eval_result
        clear_cuda()

        if not KEEP_LOCAL_FOLD_WORKDIRS:
            shutil.rmtree(local_dir, ignore_errors=True)

print("\nRequested model x fold training/evaluation complete")

In [ ]:
# ============================================================
# 7. AGGREGATE CV RESULTS + FIGURES + SELECTION GUIDANCE
# ============================================================
# Only each model's own true held-out fold is included.
# No off-diagonal 5x5 cross-evaluation matrix is used.

SUMMARY_DIR = EXP_ROOT / "summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for model_name, mcfg in MODELS.items():
    if not mcfg["enabled"]:
        continue
    for fold in FOLDS_TO_RUN:
        p = EXP_ROOT / "models" / model_name / "folds" / f"fold_{fold}" / "metrics.csv"
        if p.exists():
            rows.append(pd.read_csv(p).iloc[0].to_dict())

cv_df = pd.DataFrame(rows)
if len(cv_df) == 0:
    raise RuntimeError("No completed fold metrics were found.")
cv_df = cv_df.sort_values(["model", "fold"]).reset_index(drop=True)
cv_df.to_csv(SUMMARY_DIR / "cv_per_fold_metrics.csv", index=False)

METRIC_COLS = ["mAP50", "mAP50-95", "Precision", "Recall", "F1"]

summary_rows = []
for model_name, g in cv_df.groupby("model"):
    for m in METRIC_COLS:
        summary_rows.append({
            "model": model_name,
            "metric": m,
            "mean": g[m].mean(),
            "std": g[m].std(ddof=1),
            "min": g[m].min(),
            "max": g[m].max(),
            "n_folds": len(g),
        })
    # static architecture info (params/GFLOPs identical across folds)
    summary_rows.append({
        "model": model_name, "metric": "params",
        "mean": g["params"].iloc[0], "std": 0.0,
        "min": g["params"].iloc[0], "max": g["params"].iloc[0], "n_folds": len(g),
    })
    summary_rows.append({
        "model": model_name, "metric": "gflops",
        "mean": g["gflops"].iloc[0], "std": 0.0,
        "min": g["gflops"].iloc[0], "max": g["gflops"].iloc[0], "n_folds": len(g),
    })
    summary_rows.append({
        "model": model_name, "metric": "model_size_mb",
        "mean": g["model_size_mb"].iloc[0], "std": 0.0,
        "min": g["model_size_mb"].iloc[0], "max": g["model_size_mb"].iloc[0], "n_folds": len(g),
    })

stats_df = pd.DataFrame(summary_rows)
stats_df.to_csv(SUMMARY_DIR / "cv_summary_stats.csv", index=False)

print("\nPER-FOLD HELD-OUT RESULTS")
print("=" * 100)
print(cv_df[["model", "fold", "mAP50", "mAP50-95", "Precision", "Recall", "F1"]]
      .round(4).to_string(index=False))

print("\nMEAN +/- SD (held-out folds)")
print("=" * 100)
for model_name, g in cv_df.groupby("model"):
    print(f"\n{model_name}")
    for m in METRIC_COLS:
        print(f"  {m:10s}: {g[m].mean():.4f} +/- {g[m].std(ddof=1):.4f}")

# ------------------------------------------------------------
# Comparison table: Model | Params | GFLOPs | mAP50 | mAP50-95 | P | R | F1
# ------------------------------------------------------------
cmp_rows = []
for model_name, g in cv_df.groupby("model"):
    cmp_rows.append({
        "Model": model_name,
        "Params(M)": round(g["params"].iloc[0] / 1e6, 2),
        "GFLOPs": round(g["gflops"].iloc[0], 1),
        "Size(MB)": round(g["model_size_mb"].iloc[0], 1),
        "mAP50": round(g["mAP50"].mean(), 4),
        "mAP50-95": round(g["mAP50-95"].mean(), 4),
        "Precision": round(g["Precision"].mean(), 4),
        "Recall": round(g["Recall"].mean(), 4),
        "F1": round(g["F1"].mean(), 4),
    })
cmp_df = pd.DataFrame(cmp_rows).sort_values("mAP50", ascending=False)
cmp_df.to_csv(SUMMARY_DIR / "cv_model_comparison.csv", index=False)
print("\nCV MODEL COMPARISON (mean over held-out folds)")
print("=" * 100)
print(cmp_df.to_string(index=False))

# ------------------------------------------------------------
# Figure 1: per-model per-fold mAP50
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 5))
models = cv_df["model"].unique()
x = np.arange(N_FOLDS)
width = 0.8 / max(len(models), 1)
for i, model_name in enumerate(models):
    g = cv_df[cv_df["model"] == model_name].sort_values("fold")
    vals = [g[g["fold"] == f]["mAP50"].values[0] if (g["fold"] == f).any() else np.nan
            for f in range(1, N_FOLDS + 1)]
    ax.bar(x + (i - (len(models) - 1) / 2) * width, vals, width, label=model_name)
ax.set_xticks(x)
ax.set_xticklabels([f"Fold {f}" for f in range(1, N_FOLDS + 1)])
ax.set_ylim(0, 1)
ax.set_ylabel("mAP@50")
ax.set_title("Leak-free 5-fold CV — per-fold held-out mAP50")
ax.legend(ncol=2, fontsize=8)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(SUMMARY_DIR / "cv_per_fold_map50.png", dpi=200, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# Figure 2: mAP50 lines with mean +/- SD
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5))
for model_name, g in cv_df.groupby("model"):
    g = g.sort_values("fold")
    ax.plot(g["fold"], g["mAP50"], marker="o", linewidth=2, label=model_name)
    mu = g["mAP50"].mean()
    sd = g["mAP50"].std(ddof=1)
    ax.axhline(mu, linestyle="--", alpha=0.5)
    ax.fill_between([1, N_FOLDS], [mu - sd, mu - sd], [mu + sd, mu + sd], alpha=0.08)
ax.set_xticks(range(1, N_FOLDS + 1))
ax.set_ylim(0, 1)
ax.set_xlabel("Held-out fold")
ax.set_ylabel("mAP@50")
ax.set_title("Leak-free 5-fold CV — mAP50 per model")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(SUMMARY_DIR / "cv_map50_lines.png", dpi=200, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# Figure 3: compute cost vs accuracy (Part 19 tradeoff view)
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 6))
for _, r in cmp_df.iterrows():
    ax.scatter(r["GFLOPs"], r["mAP50"], s=120, label=None)
    ax.annotate(r["Model"], (r["GFLOPs"], r["mAP50"]),
                xytext=(6, 6), textcoords="offset points", fontsize=9)
ax.set_xlabel("GFLOPs (640x640)")
ax.set_ylabel("mAP@50 (CV mean)")
ax.set_title("Accuracy vs compute cost")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(SUMMARY_DIR / "cv_accuracy_vs_gflops.png", dpi=200, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# Automated selection guidance (not a decision, just the tradeoff math)
# ------------------------------------------------------------
print("\nSELECTION GUIDANCE (based on CV only — the independent test set is NOT used here)")
print("=" * 100)
ref = cv_df[cv_df["model"] == "yolov8n"]
if len(ref):
    ref_map50 = ref["mAP50"].mean()
    ref_g = ref["gflops"].iloc[0]
    for model_name, g in cv_df.groupby("model"):
        mu = g["mAP50"].mean()
        fl = g["gflops"].iloc[0]
        d_map = mu - ref_map50
        cost = fl / ref_g
        print(f"  {model_name:16s} mAP50 {mu:.4f}  (delta vs yolov8n {d_map:+.4f})  "
              f"compute x{cost:.2f}")

se = cv_df[cv_df["model"] == "yolov8s_lite_se"]["mAP50"]
li = cv_df[cv_df["model"] == "yolov8s_lite"]["mAP50"]
if len(se) and len(li) and "yolov8s_lite_se" in ARCH_SUMMARY and "yolov8s_lite" in ARCH_SUMMARY:
    print(f"\n  LiteSE effect: {se.mean():.4f} vs {li.mean():.4f} "
          f"({se.mean() - li.mean():+.4f}) at +{ARCH_SUMMARY['yolov8s_lite_se']['params'] - ARCH_SUMMARY['yolov8s_lite']['params']:,} params")

print("\nCV summaries and figures saved to:", SUMMARY_DIR)
print("Next: open notebook 2 (final training + independent test + ONNX).")